# Preparación de datos demográficos de OWID (2000–2022)

**Objetivo:** descargar cuatro conjuntos de datos demográficos y dos indicadores económicos/sanitarios, revisar sus columnas, preparar cada uno por separado y unirlos por `entity` y `year`. El resultado servirá para una práctica posterior de aprendizaje automático.

Ejecuta las celdas en orden. Requiere `pandas` y `matplotlib`, además de acceso a internet. Fuente: [Our World in Data](https://ourworldindata.org/).

In [0]:
import pandas as pd
import matplotlib.pyplot as plt

OPCIONES_OWID = {"User-Agent": "Our World In Data data fetch/1.0"}
BASE = "https://ourworldindata.org/grapher/"
SUFIJO = ".csv?v=1&csvType=full&useColumnShortNames=true"


## 1. Descargar las siete fuentes

Las cuatro primeras fuentes aportan los indicadores demográficos. `df` aporta gasto corriente en salud como porcentaje del PIB (`percentage_health`), `df2` aporta PIB per cápita (`gdp`) y `df_life` aporta esperanza de vida al nacer (`life_expectancy`).

In [0]:
url_population = BASE + "population-with-un-projections" + SUFIJO
url_births = BASE + "number-of-births-per-year" + SUFIJO
url_deaths = BASE + "number-of-deaths-per-year" + SUFIJO
url_child_deaths = BASE + "number-of-child-deaths-unwpp" + SUFIJO
url_health = BASE + "total-healthcare-expenditure-gdp" + SUFIJO
url_gdp = BASE + "gdp-per-capita-worldbank" + SUFIJO
url_life = BASE + "life-expectancy" + SUFIJO

df = pd.read_csv(url_health, storage_options=OPCIONES_OWID)
df2 = pd.read_csv(url_gdp, storage_options=OPCIONES_OWID)
df_life = pd.read_csv(url_life, storage_options=OPCIONES_OWID)
population = pd.read_csv(url_population, storage_options=OPCIONES_OWID)
births = pd.read_csv(url_births, storage_options=OPCIONES_OWID)
deaths = pd.read_csv(url_deaths, storage_options=OPCIONES_OWID)
child_deaths = pd.read_csv(url_child_deaths, storage_options=OPCIONES_OWID)

print("Filas y columnas:")
print("population:", population.shape)
print("births:", births.shape)
print("deaths:", deaths.shape)
print("child_deaths:", child_deaths.shape)
print("percentage_health:", df.shape)
print("gdp per capita:", df2.shape)
print("life expectancy:", df_life.shape)

## 2. Explorar las columnas

OWID puede modificar los nombres de los indicadores. Revisamos las columnas reales de las siete fuentes antes de seleccionarlas.

In [0]:
for nombre, tabla in [("population", population),
                      ("births", births),
                      ("deaths", deaths),
                      ("child_deaths", child_deaths),
                      ("percentage_health", df),
                      ("gdp", df2),
                      ("life_expectancy", df_life)]:
    print(f"\n{nombre}: {tabla.columns.tolist()}")
    display(tabla.head(3))

## 3. Preparar población

Elegimos la columna de **estimaciones históricas** para ambos sexos y todas las edades; excluimos las proyecciones. Después conservamos los años de interés.

In [0]:
columnas_population = [
    columna for columna in population.columns
        if "population" in columna and "estimates" in columna]

print("Columnas candidatas:", columnas_population)

assert len(columnas_population) == 1, "Revisar manualmente el indicador de población mostrado arriba"


col_population = columnas_population[0]

population_limpio = population[["entity", "year", col_population]].copy()

population_limpio = population_limpio.rename(columns={col_population: "population"})

population_limpio = population_limpio[population_limpio["year"].between(2000, 2022)].copy()

print(population_limpio.shape)
display(population_limpio.head())

## 4. Preparar nacimientos

Seleccionamos el número anual de nacimientos históricos. Revisamos qué entidades comparte con las otras tablas.

In [0]:
columnas_births = [c for c in births.columns
                  if "births" in c and "estimates" in c]


print("Columnas candidatas:", columnas_births)
assert len(columnas_births) == 1, "Revisar manualmente el indicador de nacimientos mostrado arriba"
col_births = columnas_births[0]

births_limpio = births[["entity", "year", col_births]].copy()
births_limpio = births_limpio.rename(columns={col_births: "births"})
births_limpio = births_limpio[births_limpio["year"].between(2000, 2022)].copy()
print(births_limpio.shape)
display(births_limpio.head())

## 5. Preparar defunciones totales

In [0]:
columnas_deaths = [c for c in deaths.columns
                   if "deaths" in c and "estimates" in c]

print("Columnas candidatas:", columnas_deaths)
assert len(columnas_deaths) == 1, "Revisar manualmente el indicador de defunciones mostrado arriba"

col_deaths = columnas_deaths[0]

deaths_limpio = deaths[["entity", "year", col_deaths]].copy()
deaths_limpio = deaths_limpio.rename(columns={col_deaths: "deaths"})
deaths_limpio = deaths_limpio[deaths_limpio["year"].between(2000, 2022)].copy()

print(deaths_limpio.shape)
display(deaths_limpio.head())

## 6. Preparar defunciones de menores de cinco años

Este indicador corresponde a niños de 0 a 4 años; no debe confundirse con mortalidad de menores de un año.

In [0]:
columnas_child = [c for c in child_deaths.columns
                  if c not in ["entity", "code", "year"]]

print("Columnas candidatas:", columnas_child)

assert len(columnas_child) == 1, "Revisar manualmente el indicador de menores de cinco años mostrado arriba"

col_child = columnas_child[0]

child_deaths_limpio = child_deaths[["entity", "year", col_child]].copy()
child_deaths_limpio = child_deaths_limpio.rename(columns={col_child: "child_deaths_under_5"})

child_deaths_limpio = child_deaths_limpio[child_deaths_limpio["year"].between(2000, 2022)].copy()

print(child_deaths_limpio.shape)
display(child_deaths_limpio.head())

## 7. Preparar gasto en salud, PIB per cápita y esperanza de vida

Seleccionamos cada indicador, lo renombramos como `percentage_health`, `gdp` y `life_expectancy`, y filtramos a 2000–2022. El PIB de OWID es per cápita (dólares internacionales constantes de 2017, ajustados por paridad de poder adquisitivo); el indicador de salud es porcentaje del PIB y la esperanza de vida corresponde al nacer.

In [0]:
columnas_health = [c for c in df.columns if c not in ["entity", "code", "year"]]
assert len(columnas_health) == 1, "Revisar la columna de gasto en salud mostrada arriba"
col_health = columnas_health[0]

health_limpio = df[["entity", "year", col_health]].copy()
health_limpio = health_limpio.rename(columns={col_health: "percentage_health"})
health_limpio = health_limpio[health_limpio["year"].between(2000, 2022)].copy()

columnas_gdp = [c for c in df2.columns
                if c not in ["entity", "code", "year", "owid_region"]]
assert len(columnas_gdp) == 1, "Revisar la columna de PIB mostrada arriba"
col_gdp = columnas_gdp[0]

gdp_limpio = df2[["entity", "year", col_gdp]].copy()
gdp_limpio = gdp_limpio.rename(columns={col_gdp: "gdp"})
gdp_limpio = gdp_limpio[gdp_limpio["year"].between(2000, 2022)].copy()

columnas_life = [c for c in df_life.columns
                 if c not in ["entity", "code", "year"]]
assert len(columnas_life) == 1, "Revisar la columna de esperanza de vida mostrada arriba"
col_life = columnas_life[0]

life_limpio = df_life[["entity", "year", col_life]].copy()
life_limpio = life_limpio.rename(columns={col_life: "life_expectancy"})
life_limpio = life_limpio[life_limpio["year"].between(2000, 2022)].copy()

for nombre, tabla in [("percentage_health", health_limpio),
                      ("gdp", gdp_limpio),
                      ("life_expectancy", life_limpio)]:
    duplicados = tabla.duplicated(["entity", "year"]).sum()
    print(nombre, "| filas:", len(tabla), "| llaves duplicadas:", duplicados)
    assert duplicados == 0, f"Llaves duplicadas en {nombre}"

display(health_limpio.head())
display(gdp_limpio.head())
display(life_limpio.head())

## 8. Revisar llaves y cobertura

Cada combinación `entity`–`year` debe aparecer una sola vez en cada tabla. `code` no se utiliza como llave porque puede faltar en agregados regionales.

In [0]:
tablas = [("population", population_limpio), ("births", births_limpio),
          ("deaths", deaths_limpio), ("child_deaths", child_deaths_limpio),
          ("percentage_health", health_limpio), ("gdp", gdp_limpio),
          ("life_expectancy", life_limpio)]

for nombre, tabla in tablas:
    duplicados = tabla.duplicated(["entity", "year"]).sum()
    print(nombre, "| entidades:", tabla["entity"].nunique(),
          "| años:", tabla["year"].min(), "a", tabla["year"].max(),
          "| llaves duplicadas:", duplicados)
    assert duplicados == 0, f"Llaves duplicadas en {nombre}"

entidades_comunes = set(population_limpio["entity"]) & set(births_limpio["entity"]) \
                  & set(deaths_limpio["entity"]) & set(child_deaths_limpio["entity"]) \
                  & set(health_limpio["entity"]) & set(gdp_limpio["entity"]) \
                  & set(life_limpio["entity"])

print("Entidades presentes en las siete fuentes:", len(entidades_comunes))
print(sorted(entidades_comunes)[:30])

## 9. Unir las siete fuentes, un paso a la vez

Usamos `inner`: permanecen únicamente las combinaciones de entidad y año presentes en ambos DataFrames. `validate="one_to_one"` detecta llaves duplicadas. Al integrar las nuevas fuentes, el resultado queda limitado a los años y entidades con datos disponibles en las siete.

In [0]:
population_births = pd.merge(
    population_limpio, births_limpio,
    on=["entity", "year"], how="inner", validate="one_to_one"
)
print("Después de población + nacimientos:", population_births.shape)
display(population_births.head())

In [0]:
population_births_deaths = pd.merge(
    population_births, deaths_limpio,
    on=["entity", "year"], how="inner", validate="one_to_one"
)
print("Después de agregar defunciones:", population_births_deaths.shape)
display(population_births_deaths.head())

In [0]:
demografia = pd.merge(
    population_births_deaths, child_deaths_limpio,
    on=["entity", "year"], how="inner", validate="one_to_one"
)
print("Dataset unido:", demografia.shape)
display(demografia.head())

In [0]:
demografia = pd.merge(
    demografia, health_limpio,
    on=["entity", "year"], how="inner", validate="one_to_one"
)
print("Después de agregar gasto en salud:", demografia.shape)

demografia = pd.merge(
    demografia, gdp_limpio,
    on=["entity", "year"], how="inner", validate="one_to_one"
)
print("Después de agregar PIB per cápita:", demografia.shape)

demografia = pd.merge(
    demografia, life_limpio,
    on=["entity", "year"], how="inner", validate="one_to_one"
)
print("Después de agregar esperanza de vida:", demografia.shape)
display(demografia.head())

## 10. Revisar faltantes y crear tasas

Eliminamos filas sin alguno de los siete indicadores necesarios. Las tasas de nacimientos y defunciones se expresan por 1,000 habitantes. `percentage_health` es gasto corriente en salud como porcentaje del PIB; `gdp` es PIB per cápita y `life_expectancy` es esperanza de vida al nacer. La razón de muertes de menores de cinco años usa nacimientos del **mismo año** como denominador; es una medida descriptiva aproximada, no una tasa de mortalidad de cohorte.

In [0]:
print("Valores faltantes antes de limpiar:")
print(demografia.isna().sum())
print("Filas antes de limpiar:", len(demografia))

demografia = demografia.dropna(subset=[
    "population", "births", "deaths", "child_deaths_under_5",
    "percentage_health", "gdp", "life_expectancy"
]).copy()
demografia = demografia[(demografia["population"] > 0) & (demografia["births"] > 0)].copy()

print("Filas después de limpiar:", len(demografia))
print("Entidades después de limpiar:", demografia["entity"].nunique())
assert not demografia.empty, "No hay entidades y años coincidentes; revisar fuentes y columnas"

In [0]:
demografia["birth_rate_per_1000"] = demografia["births"] / demografia["population"] * 1000
demografia["death_rate_per_1000"] = demografia["deaths"] / demografia["population"] * 1000
demografia["under5_deaths_per_1000_births_same_year"] = (
    demografia["child_deaths_under_5"] / demografia["births"] * 1000
)

demografia = demografia[[
    "entity", "year", "population", "births", "deaths", "child_deaths_under_5",
    "percentage_health", "gdp", "life_expectancy",
    "birth_rate_per_1000", "death_rate_per_1000",
    "under5_deaths_per_1000_births_same_year"
]].sort_values(["entity", "year"]).reset_index(drop=True)

display(demografia.head())
display(demografia.describe(include="all"))

## 11. Visualizar una entidad disponible

Puedes cambiar `entidad` por cualquier nombre de `demografia["entity"].unique()`.

In [0]:
entidad = "Mexico" if "Mexico" in demografia["entity"].values else demografia["entity"].iloc[0]
ejemplo = demografia[demografia["entity"] == entidad].sort_values("year")

ejemplo.plot(x="year", y=["birth_rate_per_1000", "death_rate_per_1000"],
             marker="o", figsize=(9, 4), title=f"Tasas demográficas: {entidad}")
plt.ylabel("Por 1,000 habitantes")
plt.grid(alpha=0.3)
plt.show()

## 12. Guardar el resultado

El CSV se guarda en `data/processed`. Contiene solamente las entidades y años con datos compartidos por las siete fuentes.

In [0]:
from pathlib import Path

carpeta_salida = Path("../data/raw")
carpeta_salida.mkdir(parents=True, exist_ok=True)

salida = carpeta_salida / "dataset_7variables_2000_2022.csv"
demografia.to_csv(salida, index=False)

print(f"Archivo guardado: {salida.resolve()} | filas: {len(demografia)}")